In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Olist Ecommerce - Customer Segmentation Analysis\n",
    "\n",
    "This notebook performs customer segmentation using RFM analysis."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "from pyspark.sql import SparkSession\n",
    "from pyspark.sql.functions import col, datediff, current_date, sum as spark_sum, avg, count, when\n",
    "from pyspark.sql.window import Window\n",
    "\n",
    "spark = SparkSession.builder \\\n",
    "    .appName(\"Customer Segmentation\") \\\n",
    "    .config(\"spark.sql.extensions\", \"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions\") \\\n",
    "    .getOrCreate()\n",
    "\n",
    "# Load fact orders\n",
    "fact_orders = spark.table(\"olist_warehouse.fact_orders_iceberg\")\n",
    "dim_customers = spark.table(\"olist_warehouse.dim_customers_iceberg\").filter(col(\"is_current\") == True)\n",
    "\n",
    "print(\"Data loaded\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. RFM Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Calculate RFM metrics\n",
    "rfm_df = fact_orders.groupBy(\"customer_sk\").agg(\n",
    "    datediff(current_date(), max(\"order_purchase_date\")).alias(\"recency\"),\n",
    "    count(\"order_id\").alias(\"frequency\"),\n",
    "    spark_sum(\"total_value\").alias(\"monetary\")\n",
    ")\n",
    "\n",
    "# Calculate percentiles for scoring\n",
    "recency_score = Window.orderBy(\"recency\")\n",
    "frequency_score = Window.orderBy(desc(\"frequency\"))\n",
    "monetary_score = Window.orderBy(desc(\"monetary\"))\n",
    "\n",
    "rfm_scored = rfm_df \\\n",
    "    .withColumn(\"r_score\", (col(\"recency\").ntile(4).over(recency_score))) \\\n",
    "    .withColumn(\"f_score\", (col(\"frequency\").ntile(4).over(frequency_score))) \\\n",
    "    .withColumn(\"m_score\", (col(\"monetary\").ntile(4).over(monetary_score))) \\\n",
    "    .withColumn(\"rfm_score\", col(\"r_score\") + col(\"f_score\") + col(\"m_score\")) \\\n",
    "    .withColumn(\"segment\",\n",
    "        when(col(\"rfm_score\") >= 10, \"Champions\")\n",
    "        .when(col(\"rfm_score\") >= 8, \"Loyal Customers\")\n",
    "        .when(col(\"rfm_score\") >= 6, \"Potential Loyalists\")\n",
    "        .when(col(\"rfm_score\") >= 4, \"At Risk\")\n",
    "        .otherwise(\"Lost\"))\n",
    "\n",
    "rfm_scored.groupBy(\"segment\").count().show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Segment Insights"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Segment metrics\n",
    "segment_metrics = rfm_scored.groupBy(\"segment\").agg(\n",
    "    avg(\"monetary\").alias(\"avg_ltv\"),\n",
    "    avg(\"frequency\").alias(\"avg_orders\"),\n",
    "    avg(\"recency\").alias(\"avg_recency_days\"),\n",
    "    count(\"*\").alias(\"customer_count\")\n",
    ").orderBy(desc(\"avg_ltv\"))\n",
    "\n",
    "segment_metrics.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Join with Customer Dimension"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "customers_segmented = dim_customers.join(rfm_scored, on=\"customer_sk\", how=\"inner\")\n",
    "\n",
    "# Update customer segment in dimension table\n",
    "for row in customers_segmented.select(\"customer_sk\", \"segment\").collect():\n",
    "    spark.sql(f\"\"\"\n",
    "        UPDATE olist_warehouse.dim_customers_iceberg \n",
    "        SET customer_segment = '{row['segment']}'\n",
    "        WHERE customer_sk = {row['customer_sk']}\n",
    "    \"\"\")\n",
    "\n",
    "print(\"Customer segments updated\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
